# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamikshaBurte/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
# Setup & Repository Init for Colab
import os, sys, subprocess

REPO_URL = "https://github.com/SamikshaBurte/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")

print("Working Directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check path"
print("Repo setup complete!")

Working Directory: /content/flyrank-ml-internship
Repo setup complete!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from google.colab import userdata

# Load dataset: try HF parquet with token, fall back to local starter CSV
try:
    hf_token = userdata.get('HF_TOKEN')
    df = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/data/train-00000-of-00001.parquet", storage_options={"token": hf_token})
    print("Successfully loaded dataset from Hugging Face Warehouse.")
except Exception as e:
    print(f"HF stream notice: {e}. Loading local starter dataset...")
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
    df['month'] = '2026-03'  # Set mid-panel month context

# Verify time window and unit of analysis grain
df_march = df[df['month'] == '2026-03'].copy()
id_col = 'url_hash' if 'url_hash' in df_march.columns else ('url' if 'url' in df_march.columns else df_march.columns[0])

print("\n=== Time Window & Grain Verification ===")
print(f"Target Month         : 2026-03")
print(f"March Row Count      : {len(df_march):,}")
print(f"Unique URLs          : {df_march[id_col].nunique():,}")
print(f"Grain Confirmed (1 row = 1 URL): {len(df_march) == df_march[id_col].nunique()}")

HF stream notice: datasets/FlyRank/internship-warehouse/data/train-00000-of-00001.parquet. Loading local starter dataset...

=== Time Window & Grain Verification ===
Target Month         : 2026-03
March Row Count      : 30,000
Unique URLs          : 30,000
Grain Confirmed (1 row = 1 URL): True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field Bucket Categorization
buckets = {
    "Features": ["impressions_90d", "ctr", "avg_position", "content_age_days"],
    "Label": ["is_declining (trend_direction == 'down')"],
    "Context": [id_col, "month"],
    "Excluded": ["clicks_90d (Excluded due to collinearity with Impressions * CTR)"]
}

print("=== Search Intelligence Field Contract ===")
for category, fields in buckets.items():
    print(f"{category.ljust(10)} : {', '.join(fields)}")

=== Search Intelligence Field Contract ===
Features   : impressions_90d, ctr, avg_position, content_age_days
Label      : is_declining (trend_direction == 'down')
Context    : content_id, month
Excluded   : clicks_90d (Excluded due to collinearity with Impressions * CTR)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 1. Verification query: Availability check
df_march['is_active'] = df_march['impressions_90d'] > 0
available_rows = df_march[df_march['is_active'] == True]
print(f"Availability Query (is_active IS TRUE): {len(available_rows):,} rows survive.\n")

# 2. Build feature matrix and target
df_march['target'] = (df_march.get('trend_direction', 'down') == 'down').astype(int)
features = ['impressions_90d', 'ctr', 'avg_position', 'content_age_days']
X = df_march[features].fillna(0)
y = df_march['target']

# 3. Deliberate Feature Leakage Experiment
X_leaked = X.copy()
X_leaked['leaked_target_column'] = df_march['target']  # Intentional leak

model_leaked = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaked, y)
model_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)

print("=== The Feature Leakage Experiment ===")
print(f"Accuracy WITH Leaked Column  : {model_leaked.score(X_leaked, y)*100:.2f}% (Falsely perfect score)")
print(f"Accuracy WITHOUT Leaked Column: {model_honest.score(X, y)*100:.2f}% (Honest baseline score)")

Availability Query (is_active IS TRUE): 30,000 rows survive.

=== The Feature Leakage Experiment ===
Accuracy WITH Leaked Column  : 100.00% (Falsely perfect score)
Accuracy WITHOUT Leaked Column: 64.46% (Honest baseline score)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Section 4 Complete ===")
print("Data limitations documented. Contract validation complete!")

=== Section 4 Complete ===
Data limitations documented. Contract validation complete!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.